
# Python Inference Tutorial - Single Model

This tutorial describes how to run an inference process using `InferPipeline` API (sync API), which is an alternative to the recommended Async API


**Requirements:**

* Run the notebook inside the Python virtual environment: ```source hailo_virtualenv/bin/activate```

When inside the ```virtualenv```, use the command ``hailo tutorial`` to open a Jupyter server that contains the tutorials.

## HEF Model Processing

In [16]:
target.release()

In [1]:
import os

# OpenCV for computer vision tasks
import cv2

# NumPy for numerical operations
import numpy as np

# PIL (Python Imaging Library) for image processing
from PIL import Image, ImageDraw, ImageFont

# Import Hailo Runtime dependencies
from hailo_platform import (
    HEF,
    ConfigureParams,
    FormatType,
    HailoSchedulingAlgorithm,
    HailoStreamInterface,
    InferVStreams,
    InputVStreamParams,
    OutputVStreamParams,
    VDevice
)

# Import Picamera2
# from picamera2 import Picamera2, Preview

In [2]:
import json
import time
import torchvision as tv


rootdir = '/home/trap-fish/uav-human-detection/hailo-ai/'
annotations_file = '/home/trap-fish/uav-human-detection/hailo-ai/shared_with_docker/visdrone/annotations_VisDroneHumans_val2.json'
images_path = '/home/trap-fish/uav-human-detection/datasets/filtered/visdrone_humans/val/images'
with open(annotations_file, 'r') as f:
    val_gt = json.load(f)
    f.close()
    
images = val_gt['images']
image_list = [(x['file_name'],x['id']) for x in images] 

print(f"frame count: {len(image_list)}")


def resize_with_letterbox(image_path, target_shape=(1,640,640,3), padding_value=(0, 0, 0)):
    """
    Resizes an image with letterboxing to fit the target size, preserving aspect ratio.
    
    Parameters:
        image_path (str): Path to the input image.
        target_shape (tuple): Target shape in NHWC format (batch_size, target_height, target_width, channels).
        padding_value (tuple): RGB values for padding (default is black padding).
        
    Returns:
        letterboxed_image (ndarray): The resized image with letterboxing.
        scale (float): Scaling ratio applied to the original image.
        pad_top (int): Padding applied to the top.
        pad_left (int): Padding applied to the left.
    """
    # Load the image from the given path
    image = cv2.imread(image_path)
    
    # Check if the image was loaded successfully
    if image is None:
        raise ValueError(f"Error: Unable to load image from path: {image_path}")
    
#     # Convert the image from BGR to RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Get the original image dimensions (height, width, channels)
    h, w, c = image.shape
    
    # Extract target height and width from target_shape (NHWC format)
    target_height, target_width = target_shape[1], target_shape[2]
    
    # Calculate the scaling factors for width and height
    scale_x = target_width / w
    scale_y = target_height / h
    
    # Choose the smaller scale factor to preserve the aspect ratio
    scale = min(scale_x, scale_y)
    
    # Calculate the new dimensions based on the scaling factor
    new_w = int(w * scale)
    new_h = int(h * scale)

    # Resize the image to the new dimensions
    resized_image = cv2.resize(image, (new_w, new_h),interpolation=cv2.INTER_LINEAR)
    
    # Create a new image with the target size, filled with the padding value
    letterboxed_image = np.full((target_height, target_width, c), padding_value, dtype=np.uint8)
    
    # Compute the position where the resized image should be placed (padding)
    pad_top = (target_height - new_h) // 2
    pad_left = (target_width - new_w) // 2
    
    # Place the resized image onto the letterbox background
    letterboxed_image[pad_top:pad_top+new_h, pad_left:pad_left+new_w] = resized_image

    
    # Return the letterboxed image, scaling ratio, and padding (top, left)
    return letterboxed_image, scale, pad_top, pad_left

def run_inference(runner, sdk_type, input_data ):
    if sdk_type == 'quantized':
        with runner.infer_context(InferenceContext.SDK_QUANTIZED) as ctx:
            output = runner.infer(ctx, input_data)
    elif sdk_type == 'native':
        with runner.infer_context(InferenceContext.SDK_NATIVE) as ctx:
            output = runner.infer(ctx, input_data)
    elif sdk_type == 'optimized':
        with runner.infer_context(InferenceContext.SDK_FP_OPTIMIZED) as ctx:
            output = runner.infer(ctx, input_data)
    else:
        print(f"Invalid sky type: {sdk_type}")
    return output  

def remove_zero_padding(output, imgid):
    # remove zero-padding and transpose detections into single array
    combined = np.empty((7, 0)) # 1 classid, xywh, score
    for i in range(output.shape[1]):
        valid_mask = np.any(output[0,i,:,:] != 0, axis=(0))  # Check if any non-zero values exist in each column
        last_valid_indices = np.argmax(~valid_mask, axis=0) # First occurrence of zero padding
        if last_valid_indices==0:
            last_valid_indices=-1
        dets = output[0, i, :, :last_valid_indices]
        class_col = np.full_like(dets[0,None], i) # add the classID into the array
        imgid_col = np.full_like(dets[0,None], imgid)
        dets = np.vstack((dets, class_col, imgid_col))
        combined = np.concatenate((combined, dets), axis=1)

    dets = combined.T
    return dets

def swap_columns(detections):
    detections[:, [0, 1,2,3]] = detections[:, [1, 0, 3, 2]]
    return detections

def ltxy2xywh(xywh):
    xywh[:,0] = xywh[:,0] # x
    xywh[:,1] = xywh[:,1] # y
    xywh[:,2] = xywh[:,2] - xywh[:,0] # x2 - x1
    xywh[:,3] = xywh[:,3] - xywh[:,1] # y2 - y1

    return xywh

def rescale_bbox(detections, w, h):
    w_arr = np.full_like(detections[:,0], w)
    h_arr = np.full_like(detections[:,0], h)
    
    detections[:,0] = detections[:,0] * w_arr
    detections[:,1] = detections[:,1] * h_arr
    detections[:,2] = detections[:,2] * w_arr
    detections[:,3] = detections[:,3] * h_arr
    
    return detections

def add_metadata_to_dict(data, data_dict):
    idx, imgid, lb_scale, lb_pad_top, lb_pad_left = data
    metadata = {
                "image_id" : imgid,
                "lbox_scale": lb_scale,
                "lbox_pad_top": lb_pad_top, 
                "lbox_pad_left": lb_pad_left
    }
    data_dict[idx]["rescale_data"] = metadata

def get_image_metadata(images_dict, img_id):
    for img in images_dict:
        if img['id'] == img_id:
            orig_w, orig_h = img['width'], img['height']
            name = img["file_name"]
            lb_scale = img["rescale_data"]["lbox_scale"]
            lb_pad_top = img["rescale_data"]["lbox_pad_top"]
            lb_pad_left = img["rescale_data"]["lbox_pad_left"]

    return orig_w, orig_h, name, lb_scale, lb_pad_top, lb_pad_left

# create a copy of the images dict so additional datapoints can be added
image_rescale_metadata = images.copy()

dataset_sz = len(image_list)
val_dataset = np.zeros((dataset_sz, 640, 640, 3))
val_imageids = np.zeros((dataset_sz,1))
for idx, imagename_id in enumerate(image_list):
    imgname, imgid = imagename_id
    image_file = os.path.join(images_path, imgname)
    if idx==dataset_sz:
        break
    img_preproc, lb_scale, lb_pad_top, lb_pad_left = resize_with_letterbox(image_file, (1,640,640,3))
    val_dataset[idx, :, :, :] = img_preproc
    val_imageids[idx] = imgid
    
    # the scaled image needs to be restored to orig size later, so these fields are saved here
    metadata = (idx, imgid, lb_scale, lb_pad_top, lb_pad_left)
    add_metadata_to_dict(metadata, image_rescale_metadata)

print(f"Created validation dataset, of shape: {val_dataset.shape}")
print(f"Created validation metadata, of length: {len(image_rescale_metadata)}")


frame count: 532
Created validation dataset, of shape: (532, 640, 640, 3)
Created validation metadata, of length: 532


In [3]:
# Load the compiled HEF to Hailo device
root_path = '/home/trap-fish/uav-human-detection/hailo-ai/shared_with_docker/visdrone/models/best/hef/'
model = 'yolov11s_visdrone_quant_exp10_singlecls_optlvl2_ds3k_816_1e5_iouthrs6_iousconf001_singlecls_bestmodel.hef'
hef_path = os.path.join(root_path, model)
hef = HEF(str(hef_path))

# Set VDevice (Virtual Device) params to disable the HailoRT service feature
params = VDevice.create_params()
params.scheduling_algorithm = HailoSchedulingAlgorithm.NONE

# Create a Hailo virtual device with the specified parameters
target = VDevice(params=params)

# Get the "network groups" (connectivity groups, aka. "different networks") information from the .hef
# Configure the device with the HEF and PCIe interface
configure_params = ConfigureParams.create_from_hef(hef=hef, interface=HailoStreamInterface.PCIe)
network_groups = target.configure(hef, configure_params)

# Select the first network group (there's only one in this case)
network_group = network_groups[0]
network_group_params = network_group.create_params()

# Create input and output virtual streams params
# These specify the format of the input and output data (in this case, 32-bit float)
input_vstreams_params = InputVStreamParams.make(network_group, format_type=FormatType.FLOAT32)
output_vstreams_params = OutputVStreamParams.make(network_group, format_type=FormatType.FLOAT32)

# Get information about the input and output virtual streams
input_vstream_info = hef.get_input_vstream_infos()[0]
output_vstream_info = hef.get_output_vstream_infos()[0]

In [4]:
def run_inference(vstream_params, input_data):
    network_group, input_vstreams_params, output_vstreams_params = vstream_params
    #input_data = {input_vstream_info.name: input_tensor}
    with InferVStreams(network_group, input_vstreams_params, output_vstreams_params) as infer_pipeline:
        with network_group.activate(network_group_params):
            infer_results = infer_pipeline.infer(input_data)
    return infer_results

In [5]:
# create an np array for the validation dataset
vstream_config = (network_group, input_vstreams_params, output_vstreams_params)
outs = []
start_time = time.perf_counter()
for idx, imagename_id in enumerate(image_list):
    img_arr = val_dataset[idx, :, :, :]
    
    # # Convert the input image to NumPy format for the model
    input_tensor_np = np.array(img_arr, dtype=np.float32)[None]
    input_tensor_np = np.ascontiguousarray(input_tensor_np)
    
    input_data = {input_vstream_info.name: input_tensor_np}

    outs.append(run_inference(vstream_config, input_data))

# Calculate and display FPS
end_time = time.perf_counter()
processing_time = end_time - start_time
fps = 1 / processing_time


In [6]:
print(f"Inference time {processing_time}\nFPS: {len(image_list) * fps}")


Inference time 16.901426230000652
FPS: 31.476633555083087


In [7]:
%reset_selective -f img_arr

In [8]:
%reset_selective -f val_dataset

In [9]:
 # add in columns for class and the imageID
def stack_image_detections(image_detections, image_id):
    stacked = np.empty((0,7))
    for idx, dets in enumerate(image_detections):
        axis_sz = dets.shape[0]
        dets_with_cls = np.full((axis_sz, 7), idx, dtype="float32")
        dets_with_cls[:, :-2] = dets
        dets_with_cls[:, -1] = image_id
        stacked = np.vstack((stacked, dets_with_cls))
    return stacked

# function below mostly copy paste from shashi on hailo community guide:
def reverse_rescale_bboxes(annotations, scale, pad_top, pad_left, original_shape):
    """
    Reverse rescales bounding boxes from the letterbox image to the original image, returning new annotations.

    Parameters:
        annotations (list of dicts): List of dictionaries, each containing a 'bbox' (x1, y1, x2, y2) and other fields.
        scale (float): The scale factor used for resizing the image.
        pad_top (int): The padding added to the top of the image.
        pad_left (int): The padding added to the left of the image.
        original_shape (tuple): The shape (height, width) of the original image before resizing.

    Returns:
        new_annotations (list of dicts): New annotations with rescaled bounding boxes adjusted back to the original image.
    """
    orig_h, orig_w = original_shape  # original image height and width
    
    new_annotations = []
    
    for annotation in annotations:
        bbox = annotation  # Bounding box as (x1, y1, x2, y2)
        
        # Reverse padding
        x1, y1, x2, y2 = bbox
        x1 -= pad_left
        y1 -= pad_top
        x2 -= pad_left
        y2 -= pad_top
        
        # Reverse scaling
        x1 = int(x1 / scale)
        y1 = int(y1 / scale)
        x2 = int(x2 / scale)
        y2 = int(y2 / scale)
        
        # Clip the bounding box to make sure it fits within the original image dimensions
        x1 = max(0, min(x1, orig_w))
        y1 = max(0, min(y1, orig_h))
        x2 = max(0, min(x2, orig_w))
        y2 = max(0, min(y2, orig_h))
        
        new_annotations.append([x1,y1, x2, y2])
        
    
    return new_annotations

def remove_zero_padding(output, imgid):
    # remove zero-padding and transpose detections into single array
    combined = np.empty((7, 0)) # 1 classid, xywh, score
    for i in range(output.shape[1]):
        valid_mask = np.any(output[0,i,:,:] != 0, axis=(0))  # Check if any non-zero values exist in each column
        last_valid_indices = np.argmax(~valid_mask, axis=0) # First occurrence of zero padding
        if last_valid_indices==0:
            last_valid_indices=-1
        dets = output[0, i, :, :last_valid_indices]
        class_col = np.full_like(dets[0,None], i) # add the classID into the array
        imgid_col = np.full_like(dets[0,None], imgid)
        dets = np.vstack((dets, class_col, imgid_col))
        combined = np.concatenate((combined, dets), axis=1)

    dets = combined.T
    return dets


def swap_columns(detections):
    swapped = detections.copy()
    swapped[:, [0, 1,2,3]] = swapped[:, [1, 0, 3, 2]]
    return swapped


def ltxy2xywh(xywh):
    coco_format = np.array(xywh)
    coco_format[:,0] = xywh[:,0] # x
    coco_format[:,1] = xywh[:,1] # y
    coco_format[:,2] = xywh[:,2] - xywh[:,0] # x2 - x1
    coco_format[:,3] = xywh[:,3] - xywh[:,1] # y2 - y1

    return coco_format

def rescale_bbox(detections, w, h):
    w_arr = np.full_like(detections[:,0], w)
    h_arr = np.full_like(detections[:,0], h)
    scaled = np.zeros_like(detections)
    
    scaled[:,0] = detections[:,0] * w_arr
    scaled[:,1] = detections[:,1] * h_arr
    scaled[:,2] = detections[:,2] * w_arr
    scaled[:,3] = detections[:,3] * h_arr
    
    scaled[:, 4:] = detections[:, 4:]
    
    return scaled

def add_metadata_to_dict(data, data_dict):
    idx, imgid, lb_scale, lb_pad_top, lb_pad_left = data
    metadata = {
                "image_id" : imgid,
                "lbox_scale": lb_scale,
                "lbox_pad_top": lb_pad_top, 
                "lbox_pad_left": lb_pad_left
    }


    data_dict[idx]["rescale_data"] = metadata
    


def get_image_metadata(images_dict, img_id):
    for img in images_dict:
        if img['id'] == img_id:
            orig_w, orig_h = img['width'], img['height']
            name = img["file_name"]
            lb_scale = img["rescale_data"]["lbox_scale"]
            lb_pad_top = img["rescale_data"]["lbox_pad_top"]
            lb_pad_left = img["rescale_data"]["lbox_pad_left"]

    return orig_w, orig_h, name, lb_scale, lb_pad_top, lb_pad_left

def batch_detections_to_json(detections):
    class_remap = {0: 1, 1: 2}
    res = []
    annId = 1
    for el in detections:
        x,y,w,h,conf,classid,imgid = el
        imgid = int(imgid)
        res.append({"image_id": imgid, "category_id": class_remap[classid], "bbox": [x,y,w,h], "score": conf, "id": annId, "segmentation": []}) 
        annId +=1
    return res



In [10]:
import json
import numpy as np
import os

# Create output file and write initial bracket
output_path = '/home/trap-fish/uav-human-detection/hailo-ai/shared_with_docker/visdrone/results2'
filename = model.replace('hef', 'json')
output_file = os.path.join(output_path, filename)
with open(output_file, 'w') as f:
    f.write("[\n")

annId = 1
first = True  # To manage commas between JSON objects

for idx, output in enumerate(outs):  # process one image at a time
    key = next(iter(output))  # assumes one key per dict
    detections = output[key]
    if detections == [] or idx==267:
        print(idx)
        continue
    single_img_det = stack_image_detections(detections[0], idx + 1)
    xyxy_dets = swap_columns(single_img_det)
    dets_scaled = rescale_bbox(xyxy_dets, 640, 640)

    w, h, name, lb_scale, lb_pad_top, lb_pad_left = get_image_metadata(image_rescale_metadata, idx + 1)
    xyxy_orig = reverse_rescale_bboxes(dets_scaled[:, :4], lb_scale, lb_pad_top, lb_pad_left, (h, w))

    dets_scaled[:, :4] = ltxy2xywh(np.array(xyxy_orig))

    for el in dets_scaled:
        x, y, w, h, conf, classid, imgid = el
        imgid = int(imgid)
        category_id = {0: 1, 1: 2}.get(int(classid), 1)

        det_json = {
            "image_id": imgid,
            "category_id": category_id,
            "bbox": [x, y, w, h],
            "score": float(conf),
            "id": annId,
            "segmentation": []
        }

        # Append to file
        with open(output_file, 'a') as f:
            if not first:
                f.write(",\n")
            json.dump(det_json, f)
            first = False

        annId += 1

# Finalize the JSON array
with open(output_file, 'a') as f:
    f.write("\n]\n")


267


In [11]:
target.release()

In [12]:

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import pylab

import json
import os

pylab.rcParams['figure.figsize'] = (10.0, 8.0)

annType = ['segm','bbox','keypoints']
annType = annType[1]      #specify type here
prefix = 'person_keypoints' if annType=='keypoints' else 'instances'
print('Running demo for *%s* results.'%(annType))

#initialize COCO ground truth api
rootdir = '/home/trap-fish/uav-human-detection/hailo-ai/'
# annFile = os.path.join(rootdir, 'shared_with_docker/visdrone/annotations_VisDroneHumans_val.json')
annFile = '/home/trap-fish/uav-human-detection/datasets/filtered/visdrone_humans/val/export_annotations_VisDroneHumans_val.json'
cocoGt=COCO(annFile)

#initialize COCO detections api
# resFile= outputfile
resFile = output_file
cocoDt=cocoGt.loadRes(resFile)

imgIds=sorted(cocoGt.getImgIds())
imgIds = imgIds[0:]
catIDs = [1,2] # 81 for all IDs
useCats = 0
maxDets = [1, 10, 100]

# running evaluation
cocoEval = COCOeval(cocoGt,cocoDt,annType)
cocoEval.params.imgIds  = imgIds
cocoEval.params.catIds = catIDs
cocoEval.params.maxDets = maxDets
cocoEval.params.useCats = useCats
cocoEval.evaluate()
cocoEval.accumulate()
cocoEval.summarize()


Running demo for *bbox* results.
loading annotations into memory...
Done (t=0.23s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.34s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=23.28s).
Accumulating evaluation results...
DONE (t=0.51s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.215
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.519
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.142
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.184
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.427
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.671
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.022
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.146
 Average Recall     (AR) @[ IoU

In [13]:
import faster_coco_eval

# Replace pycocotools with faster_coco_eval
faster_coco_eval.init_as_pycocotools()

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

anno = COCO(str(annFile))  # init annotations api
# resFile = "/home/trap-fish/uav-human-detection/hailo-ai/shared_with_docker/visdrone/results/yolov5sp2_openvino_model.json"
pred = anno.loadRes(str(resFile))  # init predictions api (must pass string, not Path)

val = COCOeval(anno, pred, "bbox")
val.params.imgIds  = imgIds
val.params.maxDets = maxDets
val.params.catIds = catIDs
val.params.useCats = useCats


val.evaluate()
val.accumulate()
val.summarize()


Evaluate annotation type *bbox*
COCOeval_opt.evaluate() finished...
DONE (t=0.68s).
Accumulating evaluation results...
COCOeval_opt.accumulate() finished...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.215
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.519
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.142
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.184
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.427
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.671
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.022
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.146
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.300
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.273
 Average Recall     (AR) @[

In [ ]:
import cv2
import matplotlib.pyplot as plt
import json
def plot_annotations(root_path, annotations, image_id, categories):
    for image in images:
        if image["id"] == image_id:
            filename = image["file_name"]
    image_path = os.path.join(root_path, filename)
    
    # Load image
    image = cv2.imread(image_path)
    if image is None:
        print("Warning: Image not loaded")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Filter annotations for the specific image_id
    image_annotations = [ann for ann in annotations if ann['image_id'] == image_id]
    
    # Draw bounding boxes
    for ann in image_annotations:
        if ann["category_id"] in categories:
            x, y, w, h = map(int, ann['bbox'])
            cv2.rectangle(image, (x, y), (x + w, y + h), (255, 0, 0), 1)
    
    # Display the image
    plt.figure(figsize=(10, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.show()

# Example usage
predfile = outputfile
with open(predfile, 'r') as f:
    preds = json.load(f)
    f.close()

openvino_preds = '/home/trap-fish/uav-human-detection/hailo-ai/shared_with_docker/visdrone/results/yolov5n_openvino_model.json'
with open(openvino_preds, 'r') as f:
    ov_preds = json.load(f)
    f.close()

imagedir = '/home/trap-fish/uav-human-detection/datasets/filtered/visdrone_humans/val/images'

# plot_annotations(imagedir, ov_preds, 4, (1,2))
plot_annotations(imagedir, preds, 32, (1,2))

In [ ]:
rootdir = '/home/trap-fish/uav-human-detection/datasets/filtered/visdrone_humans/val/images/'
gt_filepath = '/home/trap-fish/uav-human-detection/datasets/filtered/visdrone_humans/val/export_annotations_VisDroneHumans_val.json'
with open(gt_filepath, 'r') as f:
    gt_anns = json.load(f)
    f.close()

gt_anns = gt_anns["annotations"]
plot_annotations(rootdir, gt_anns, 32, (1,2))

In [ ]:
from picamera2 import Picamera2, Preview
import cv2
import time
picam2 = Picamera2()

print("Available sensor modes:")
for i, mode in enumerate(picam2.sensor_modes):
    print(f"Mode {i}: {mode['size']}x{mode['fps']}fps")

# Choose a mode (let's say we want the second mode, index 1)
chosen_mode = picam2.sensor_modes[0]

# Create a configuration using the chosen mode
config = picam2.create_preview_configuration(
    main={
        "size": chosen_mode["size"],
        "format": "RGB888"
    },
    controls={
        "FrameRate": chosen_mode["fps"]
    },
    sensor={
        "output_size": chosen_mode["size"],
        "bit_depth": chosen_mode["bit_depth"]
    }
)


camera_config = picam2.create_preview_configuration()
picam2.configure(camera_config)
picam2.start()
frame = picam2.capture_array()

In [ ]:

camera_config = picam2.create_preview_configuration()
picam2.configure(camera_config)
picam2.start()
frame = picam2.capture_array()

### Testing inference on camera stream

In [ ]:
from picamera2 import Picamera2, Preview
import cv2
import time


picam2 = Picamera2()

print("Available sensor modes:")
for i, mode in enumerate(picam2.sensor_modes):
    print(f"Mode {i}: {mode['size']}x{mode['fps']}fps")

# Choose a mode (let's say we want the second mode, index 1)
chosen_mode = picam2.sensor_modes[0]

# Create a configuration using the chosen mode
config = picam2.create_preview_configuration(
    main={
        "size": chosen_mode["size"],
        "format": "RGB888"
    },
    controls={
        "FrameRate": chosen_mode["fps"]
    },
    sensor={
        "output_size": chosen_mode["size"],
        "bit_depth": chosen_mode["bit_depth"]
    }
)


# camera_config = picam2.create_preview_configuration()
# picam2.configure(camera_config)
# picam2.start()
# frame = picam2.capture_array()

# # create an np array for the validation dataset
# vstream_config = (network_group, input_vstreams_params, output_vstreams_params)
# outs = []
# start_time = time.perf_counter()
# for idx, imagename_id in enumerate(image_list):
#     img_arr = val_dataset[idx, :, :, :]
    
#     # # Convert the input image to NumPy format for the model
#     input_tensor_np = np.array(img_arr, dtype=np.float32)[None]
#     input_tensor_np = np.ascontiguousarray(input_tensor_np)
    
#     input_data = {input_vstream_info.name: input_tensor_np}

#     outs.append(run_inference(vstream_config, input_data))

# # Calculate and display FPS
# end_time = time.perf_counter()
# processing_time = end_time - start_time
# fps = 1 / processing_time

camera_config = picam2.create_preview_configuration()
picam2.configure(camera_config)
picam2.start()
frame = picam2.capture_array()
picam2.close()

In [ ]:
frame.shape

In [ ]:
from picamera2 import Picamera2, Preview
import cv2
import time

vstream_config = (network_group, input_vstreams_params, output_vstreams_params)

picam2 = Picamera2()

# Choose a mode (let's say we want the second mode, index 1)
chosen_mode = picam2.sensor_modes[0]

# Create a configuration using the chosen mode
config = picam2.create_preview_configuration(
    main={
        "size": chosen_mode["size"],
        "format": "RGB888"
    },
    controls={
        "FrameRate": chosen_mode["fps"]
    },
    sensor={
        "output_size": chosen_mode["size"],
        "bit_depth": chosen_mode["bit_depth"]
    }
)



camera_config = picam2.create_preview_configuration()
picam2.configure(camera_config)
picam2.start()


print("Capturing video...")
i=0
start_time = time.perf_counter()
frame_count = 300
outs = []
while i < frame_count:
    frame = picam2.capture_array()
    i+=1
    example_resz, _ = resize_and_pad(frame[:,:,:-1], (640, 640))
    input_img_np = example_resz
    
    # Resize and pad the sample image to the desired input size, retrieving transformation data.
    #input_img_np, transform_data = resize_and_pad(frame, input_vstream_info.shape[::-1][1:], True)
    
    # Convert the input image to NumPy format for the model
    input_tensor_np = np.array(input_img_np, dtype=np.float32)[None]
    input_tensor_np = np.ascontiguousarray(input_tensor_np)
    
    # Run inference
    input_data = {input_vstream_info.name: input_tensor_np}
    with InferVStreams(network_group, input_vstreams_params, output_vstreams_params) as infer_pipeline:
        with network_group.activate(network_group_params):
            infer_results = infer_pipeline.infer(input_data)
    
    outs.append(infer_results)
    
        
end_time = time.perf_counter()
picam2.close()

# Calculate and display FPS
processing_time = end_time - start_time
fps = 1 / processing_time

In [ ]:
picam2.close()
frame_count / processing_time

In [ ]:
len(outs)

In [ ]:
from typing import Tuple, Optional, NamedTuple
class ImageTransformData(NamedTuple):
    """
    A data class that stores transformation information applied to an image.

    Attributes:
        offset (Tuple[int, int]): The (x, y) offset where the resized image was pasted.
        scale (float): The scaling factor applied to the original image.
    """
    offset: Tuple[int, int]
    scale: float
    
def resize_and_pad(
    image: np.ndarray,
    target_sz: Tuple[int, int],
    return_transform_data: bool = False,
    fill_color: Tuple[int, int, int] = (255, 255, 255)
) -> Tuple[np.ndarray, Optional[ImageTransformData]]:
    """
    Resize an image while maintaining its aspect ratio and pad it to fit the target size.

    Args:
        image (np.ndarray): The original image as a numpy array.
        target_sz (Tuple[int, int]): The desired size (width, height) for the output image.
        return_transform_data (bool, optional): If True, returns transformation data (offset and scale).
        fill_color (Tuple[int, int, int], optional): The color to use for padding (default is white).

    Returns:
        Tuple[np.ndarray, Optional[ImageTransformData]]: The resized and padded image,
        and optionally the transformation data.
    """
    target_width, target_height = target_sz
    orig_height, orig_width = image.shape[:2]
    
    aspect_ratio = orig_width / orig_height
    target_aspect_ratio = target_width / target_height

    if aspect_ratio > target_aspect_ratio:
        new_width = target_width
        new_height = int(new_width / aspect_ratio)
        scale = target_width / orig_width
    else:
        new_height = target_height
        new_width = int(new_height * aspect_ratio)
        scale = target_height / orig_height

    resized_image = cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_NEAREST)

    paste_x = (target_width - new_width) // 2
    paste_y = (target_height - new_height) // 2

    padded_image = np.full((target_height, target_width, 3), fill_color, dtype=np.uint8)
    padded_image[paste_y:paste_y+new_height, paste_x:paste_x+new_width] = resized_image

    if return_transform_data:
        transform_data = ImageTransformData(offset=(paste_x, paste_y), scale=scale)
        return padded_image, transform_data
    else:
        return padded_image, None

In [ ]:
from PIL import Image

picam2.start()
print("Capturing video...")
start_time = time.perf_counter()
frame = picam2.capture_array()
example_resz, _ = resize_and_pad(frame[:,:,:-1], (640, 640))
print(f"frame shape: {example_resz.shape}")

input_img_np = example_resz

# Resize and pad the sample image to the desired input size, retrieving transformation data.
#input_img_np, transform_data = resize_and_pad(frame, input_vstream_info.shape[::-1][1:], True)

# Convert the input image to NumPy format for the model
input_tensor_np = np.array(input_img_np, dtype=np.float32)[None]
input_tensor_np = np.ascontiguousarray(input_tensor_np)

# Run inference
input_data = {input_vstream_info.name: input_tensor_np}
with InferVStreams(network_group, input_vstreams_params, output_vstreams_params) as infer_pipeline:
    with network_group.activate(network_group_params):
        infer_results = infer_pipeline.infer(input_data)

# Transpose and extract the first element of the quantized results
outputs = infer_results[output_vstream_info.name]#.transpose(0, 1, 3, 2)[0]

In [ ]:
outputs

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pycocotools.coco import COCO

# === Path to your annotation file ===
coco_annotation_path = "/home/trap-fish/uav-human-detection/hailo-ai/shared_with_docker/visdrone/annotations_VisDroneHumans_val.json"


# Load the dataset
coco = COCO(coco_annotation_path)

# Get all annotations
annotations = coco.loadAnns(coco.getAnnIds())

# Extract width, height, and compute area
object_data = []

for ann in annotations:
    x, y, width, height = ann['bbox']
    area = width * height
    object_data.append({'width': width, 'height': height, 'area': area})

# Convert to DataFrame
df = pd.DataFrame(object_data)

# === Save CSV ===
df.to_csv("object_sizes.csv", index=False)
print("Saved object size data to object_sizes.csv")

# === Plotting ===
plt.figure(figsize=(15, 5))

# Area histogram
plt.subplot(1, 3, 1)
plt.hist(df['area'], bins=5, color='skyblue', edgecolor='black')
plt.title("Object Area Distribution")
plt.xlabel("Area (pixels^2)")
plt.ylabel("Frequency")

# Width histogram
plt.subplot(1, 3, 2)
plt.hist(df['width'], bins=5, color='salmon', edgecolor='black')
plt.title("Object Width Distribution")
plt.xlabel("Width (pixels)")
plt.ylabel("Frequency")

# Height histogram
plt.subplot(1, 3, 3)
plt.hist(df['height'], bins=5, color='limegreen', edgecolor='black')
plt.title("Object Height Distribution")
plt.xlabel("Height (pixels)")
plt.ylabel("Frequency")

plt.tight_layout()
plt.savefig("object_size_distribution.png")
plt.show()

print("Saved histogram as object_size_distribution.png")
